# syft-ingest API Exploration

In [ ]:
from pathlib import Path

import syft_ingest as si

In [ ]:
notebook_dir = Path.cwd()
notebook_dir

## 1. Simplified gather() API — YouTube video

In [ ]:
# Simplest case: just platform + URLs
corpus = si.gather(
    "youtube", ["https://www.youtube.com/watch?v=zY2dAK-pMPI&t=11s"]
)  # Andrew Trask: talk

print(f"Items fetched: {len(corpus.all_items())}")
if corpus.all_items():
    item = corpus.all_items()[0]
    print(f"  Title: {item.title}")
    print(f"  Author: {item.author}")

In [ ]:
corpus.export(notebook_dir / "data" / "youtube.jsonl")

## 2. Youtube Channel 

In [ ]:
# With config options passed as kwargs
corpus = si.gather(
    "youtube",
    ["https://www.youtube.com/@iamtrask"],
    playlistend=3,  # Cap at 3 videos to keep it fast
)

print(f"Videos found: {len(corpus.all_items())}")
for item in corpus.all_items():
    print(f"  [{item.published_at}] {item.title}")

In [ ]:
corpus.export(notebook_dir / "data" / "youtube2.jsonl")

## 3. Facebook scraping

Requires `BRIGHTDATA_API_TOKEN`. Triggers a real scrape job and polls until done (~60-120s).

In [ ]:
import os

token = os.getenv("BRIGHTDATA_API_TOKEN")
print(
    f"BRIGHTDATA_API_TOKEN : {token[:3] + '...' + token[-3:] if token else 'NOT SET -- BrightData cells will fail'}"
)
if not token:
    raise ValueError("BRIGHTDATA_API_TOKEN environment variable is not set.")

In [ ]:
corpus = await si.async_gather(
    "facebook",
    ["https://www.facebook.com/profile.php?id=61583734012155"],
    author="Syft Influencer Test",
    posts_limit=3,
)

print(f"Items fetched: {len(corpus.all_items())}")
print(f"Author: {corpus.person}")

if corpus.all_items():
    item = corpus.all_items()[0]
    print("\nFirst item:")
    print(f"  Type: {type(item).__name__}")
    print(f"  Title: {item.title}")
    print(f"  Text: {(item.text or '')[:120]}")

In [ ]:
corpus.export(notebook_dir / "data" / "fb.jsonl")

## 4. Instagram scraping with posts_limit for testing

In [ ]:
corpus = await si.async_gather(
    "instagram",
    ["https://www.instagram.com/syft_influencer_test/"],
    author="Syft Influencer Test",
    posts_limit=3,
    timeout=120,
    poll_interval=5,
)

print(f"Items fetched: {len(corpus.all_items())}")
for i, item in enumerate(corpus.all_items()[:3], 1):
    print(f"\n{i}. {type(item).__name__}: {item.title}")

In [ ]:
corpus.export(notebook_dir / "data" / "ig.jsonl")

## 5. Concurrent fetching — async power

Run YouTube, Facebook, and Instagram **simultaneously**. Total time = slowest scrape, not the sum.

In [ ]:
import asyncio
import time

start = time.time()

# All three run concurrently — event loop drives all poll loops simultaneously
corpus_yt, corpus_fb, corpus_ig = await asyncio.gather(
    si.async_gather("youtube", ["https://www.youtube.com/@iamtrask"], playlistend=3),
    si.async_gather(
        "facebook",
        ["https://www.facebook.com/profile.php?id=61583734012155"],
        author="Syft Influencer Test",
        posts_limit=3,
        timeout=300,
    ),
    si.async_gather(
        "instagram",
        ["https://www.instagram.com/syft_influencer_test/"],
        author="Syft Influencer Test",
        posts_limit=3,
        timeout=300,
    ),
)

elapsed = time.time() - start

print(f"YouTube:   {len(corpus_yt.all_items())} items")
print(f"Facebook:  {len(corpus_fb.all_items())} items")
print(f"Instagram: {len(corpus_ig.all_items())} items")
print(f"\nTotal time: {elapsed:.1f}s (sequential would be ~{elapsed * 2:.0f}s+)")

In [ ]:
corpus_yt.export(notebook_dir / "data" / "youtube_async.jsonl")
corpus_fb.export(notebook_dir / "data" / "facebook_async.jsonl")
corpus_ig.export(notebook_dir / "data" / "instagram_async.jsonl")

## 7. Error handling and validation

In [ ]:
# Invalid platform raises ValueError
try:
    corpus = si.gather("tiktok", ["https://tiktok.com/user"])
except KeyError as e:
    print(f"✅ ValueError caught for unsupported platform: {e}")

# Missing URLs raises ValueError
try:
    corpus = si.gather("youtube", None)
except ValueError as e:
    print(f"✅ ValueError caught for missing URLs: {e}")